In [102]:
import ast

import polars as pl

In [103]:
df = pl.read_csv("../data.csv", separator=";")
df.shape

(1007, 11)

In [104]:
df.columns

['id',
 'projektname',
 'kurzzusammenfassung',
 'art',
 'einsatzbereich',
 'status',
 'organisation',
 'webseite_link',
 'quelle',
 'lizenz',
 'lizenz_organisation']

## Listen-Spalten parsen

In [105]:
df = df.with_columns(
    pl.col("art").map_elements(ast.literal_eval, return_dtype=pl.List(pl.Utf8)),
    pl.col("einsatzbereich").map_elements(ast.literal_eval, return_dtype=pl.List(pl.Utf8)),
)
df.select("projektname", "art", "einsatzbereich").head()

projektname,art,einsatzbereich
str,list[str],list[str]
"""Qualitätsanalyse von OpenStree…","[""Webanwendungen"", ""Datenreporting"", ""Datenanalyse""]","[""Stadtentwicklung"", ""Gesundheit"", ""Klima & Umwelt""]"
"""Mithilfe von KI Wirkungsmessun…","[""Künstliche Intelligenz"", ""Automatisierung"", … ""Datenanalyse""]","[""Anti Dismkriminierung"", ""Soziale Dienste"", … ""Jugendhilfe""]"
"""Wertvolle Zeit sparen und Fehl…","[""Datenreporting"", ""Automatisierung""]","[""Bildung""]"
"""Automatisiertes Monitoring von…","[""Wirkungsmessung"", ""Automatisierung""]","[""Bildung"", ""Jugendhilfe""]"
"""Automatisierte Fragebogenauswe…","[""Künstliche Intelligenz"", ""Automatisierung"", … ""Datenanalyse""]","[""Stadtentwicklung"", ""Soziale Dienste"", … ""Inklusion & Teilhabe""]"


## Häufigste Kategorien (`art`)


In [106]:
(
    df.select("art")
    .explode("art")
    .group_by("art")
    .len()
    .sort("len", descending=True)
)

/tmp/ipykernel_134609/3849583773.py:3: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("art")


art,len
str,u32
"""Webanwendungen""",459
"""Künstliche Intelligenz""",389
"""Datenreporting""",322
"""Datenanalyse""",289
"""Öffentliche Daten""",197
…,…
"""Wissensorganisation""",22
"""Virtuelle Assistenz""",20
"""Recomender System""",14


## Häufigste Einsatzbereiche

In [110]:
(
    df.select("einsatzbereich")
    .explode("einsatzbereich")
    .group_by("einsatzbereich")
    .len()
    .sort("len", descending=True)
)

/tmp/ipykernel_134609/399859936.py:3: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("einsatzbereich")


einsatzbereich,len
str,u32
"""Inklusion & Teilhabe""",361
"""Klima & Umwelt""",263
"""Soziale Dienste""",232
"""Stadtentwicklung""",228
"""Bildung""",192
…,…
null,62
"""Internationale Projekte""",52
"""Flucht & Migration""",38


## Verteilung der (normalisierten) Status-Werte

In [111]:
df["status"].value_counts().sort("count", descending=True)

status,count
str,u32
"""In Betrieb""",440
"""Unbekannt""",367
"""Abgeschlossen""",166
"""In Weiterentwicklung""",21
"""Im Testbetrieb""",6
"""Eingestellt""",4
"""In Planung""",2
"""Prototyp""",1


## Beispiel-Filter: alle Projekte einer Kategorie

Zeilen, deren `art`-Liste die Kategorie `Datenanalyse` enthält.

In [112]:
(
    df.filter(pl.col("art").list.contains("Datenanalyse"))
    .select("projektname", "status", "organisation")
    .head()
)

projektname,status,organisation
str,str,str
"""Qualitätsanalyse von OpenStree…","""Abgeschlossen""","""a tip: tap e.V., CorrelAid e.V…"
"""Mithilfe von KI Wirkungsmessun…","""Abgeschlossen""","""In safe hands e.V., CorrelAid …"
"""Automatisierte Fragebogenauswe…","""Abgeschlossen""","""Babylotse, CorrelAid e.V."""
"""Automatisiertes Qualitätsmanag…","""In Betrieb""","""Sindbad, CorrelAid e.V."""
"""Auswertung einer Umfrage zu KI…","""Abgeschlossen""","""Civic Data Lab, lagfa bayern, …"
